# Predicao em Escala - EfficientNet-B0 (timm)

Notebook para inferencia em batch sobre imagens e frames de video.

**Funcionalidades:**
- Carregamento do modelo EfficientNet-B0 pre-treinado (timm)
- Pre-processamento padrao ImageNet
- `predicao_batch`: inferencia em lotes sobre lista de imagens PIL
- `obter_frames_video`: extrai frames de video via ffmpeg com taxa configuravel
- `obter_imagens_diretorio`: lista imagens de um diretorio com seus caminhos

**Dependencias:**
```
# ffmpeg deve estar instalado no sistema:
# Ubuntu/Debian: sudo apt install ffmpeg
# macOS:         brew install ffmpeg
# Windows:       https://ffmpeg.org/download.html e executável deve estar no path
```

#### Imports e verificacao do ambiente

In [1]:
import os
import subprocess
import shutil
import json
import math
import time
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
from PIL import Image
import torch
import torchvision.transforms as T
import timm
from tqdm.auto import tqdm

print(f"PyTorch  : {torch.__version__}")
print(f"timm     : {timm.__version__}")
print(f"Device   : {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

# Verificar ffmpeg
_ffmpeg = shutil.which("ffmpeg")
_ffprobe = shutil.which("ffprobe")
print(f"ffmpeg   : {_ffmpeg if _ffmpeg else 'NAO ENCONTRADO - instalar para usar obter_frames_video'}")
print(f"ffprobe  : {_ffprobe if _ffprobe else 'NAO ENCONTRADO'}")


PyTorch  : 2.11.0+cpu
timm     : 1.0.26
Device   : cpu
ffmpeg   : C:\ProgramData\chocolatey\bin\ffmpeg.EXE
ffprobe  : C:\ProgramData\chocolatey\bin\ffprobe.EXE


#### Modelo e transformacoes

In [2]:
# carregando modelo - na primeira vez, sera feito download dos pesos

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "efficientnet_b0"

# Carrega do modelo ( pretrained=True: pesos ImageNet-1k )
print(f"Carregando {MODEL_NAME}...")
model_temp = timm.create_model(MODEL_NAME, pretrained=True)
print(f"Modelo carregado.")


Carregando efficientnet_b0...


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

Modelo carregado.


C:\Users\gusta\IdeaProjects\mba-ia\05-visao-computacional\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gusta\.cache\huggingface\hub\models--timm--efficientnet_b0.ra_in1k. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
# agora vamos salvar os pesos na pasta corrente
PATH_PESOS_EFFB0 = "./efficientnet_b0.pth"
torch.save(model_temp.state_dict(), PATH_PESOS_EFFB0)
model_temp = None

In [4]:
model_effb0 = timm.create_model(MODEL_NAME, pretrained=True)
model_effb0.eval()
model_effb0 = model_effb0.to(DEVICE)


In [5]:
# agora vamos carregar o modelo com os pesos salvos na pasta corrente.

# Carrega do modelo ( pretrained=False)
print(f"Carregando {MODEL_NAME}...")
model_effb0 = timm.create_model(MODEL_NAME, pretrained=False)

state_dict = torch.load(PATH_PESOS_EFFB0, map_location=DEVICE)
model_effb0.load_state_dict(state_dict)
print("Carregou modelo usando pesos locais")

Carregando efficientnet_b0...
Carregou modelo usando pesos locais


In [6]:
# preparando para inferência

model_effb0.eval()
model_effb0 = model_effb0.to(DEVICE)

n_params = sum(p.numel() for p in model_effb0.parameters())
print(f"Parametros : {n_params:,}")
print(f"Device     : {next(model_effb0.parameters()).device}")

# Configuracao de pre-processamento via timm 
# timm expoe os dados de config do modelo treinado (mean, std, input_size)
data_cfg = timm.data.resolve_model_data_config(model_effb0)
print()
print(f"Config do modelo:")
print(f"  input_size : {data_cfg['input_size']}")
print(f"  mean       : {data_cfg['mean']}")
print(f"  std        : {data_cfg['std']}")
print(f"  interpolation: {data_cfg['interpolation']}")


Parametros : 5,288,548
Device     : cpu

Config do modelo:
  input_size : (3, 224, 224)
  mean       : (0.485, 0.456, 0.406)
  std        : (0.229, 0.224, 0.225)
  interpolation: bicubic


In [7]:
# Transformacoes para inferencia (sem augmentation)
# define o pre-processamento das imagens antes de enviar para predicao do modelo

# O timm facilita bastanet essa etapa 
transform_inference = timm.data.create_transform(
    **data_cfg,
    is_training=False   # sem RandomCrop, RandomFlip, etc.
)

# Se fóssemos fazer isso manualmente, o trabalho seria muito maior
_, h, w = data_cfg["input_size"]   # (C, H, W) -> pega H e W
interp_map = {"bicubic": T.InterpolationMode.BICUBIC,
              "bilinear": T.InterpolationMode.BILINEAR,
              "nearest": T.InterpolationMode.NEAREST}
interp_mode = interp_map.get(data_cfg["interpolation"], T.InterpolationMode.BICUBIC)

transform_manual = T.Compose([
    T.Resize((h, w), interpolation=interp_mode),  # redimensiona para 224x224
    T.ToTensor(),                                  # [0,255] uint8 -> [0,1] float32
    T.Normalize(
        mean=list(data_cfg["mean"]),
        std=list(data_cfg["std"])
    ),
])

print("Transformacoes de inferencia definidas:")
print(transform_inference)

Transformacoes de inferencia definidas:
Compose(
    Resize(size=256, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeToTensor()
    Normalize(mean=tensor([0.4850, 0.4560, 0.4060]), std=tensor([0.2290, 0.2240, 0.2250]))
)


#### predicao em escala (em lote) 

In [8]:
# em geral, especialmente usando GPUs, se você tiver 50 imagens e fizer
# a predicao uma por uma (50 predicoes variando a imagem) é ineficiente
# >>> É preferível fazer 1 prediçao de 50 imagens

def predicao_batch(model, lst_imagens, transform, batch_size=32, 
                   device=None, top_k=5, verbose=True):
    # predicao em batch sobre uma lista de imagens PIL.
    # retorna para cada imagem um dicionario com
    #        'logits'     
    #        'probas'     
    #        'top_k_idx'  
    #        'top_k_proba'

    if device is None: device = next(model.parameters()).device
    model.eval()
    resultados = []
    n_total = len(lst_imagens)
    n_batches = math.ceil(n_total / batch_size)
    pbar = tqdm(total=n_total, desc="Predicao", unit="img", disable=not verbose)
    with torch.no_grad():
        for i in range(n_batches):
            inicio = i * batch_size
            fim = min(inicio + batch_size, n_total)
            batch_imgs = lst_imagens[inicio:fim]
            # Converter para RGB e aplicar transformacoes
            tensores = []
            for img in batch_imgs:
                if img.mode != "RGB": img = img.convert("RGB")
                tensores.append(transform(img))

            # Empilhar em um unico tensor (B, C, H, W)
            batch_tensor = torch.stack(tensores).to(device)
            # Forward pass
            logits = model(batch_tensor)            # (B, n_classes)
            probas = torch.softmax(logits, dim=1)   # (B, n_classes)
            logits_np = logits.cpu().numpy()
            probas_np = probas.cpu().numpy()
            for j in range(len(batch_imgs)):
                top_idx = np.argsort(probas_np[j])[::-1][:top_k]
                resultados.append({
                    "logits"      : logits_np[j],
                    "probas"      : probas_np[j],
                    "top_k_idx"   : top_idx,
                    "top_k_proba" : probas_np[j][top_idx],
                })

            pbar.update(len(batch_imgs))

    pbar.close()
    return resultados



#### obter_frames_video — extrai frames via ffmpeg

In [14]:
def _duracao_video(video_path):
    # Retorna a duracao do video em segundos usando ffprobe.
    ffprobe = shutil.which("ffprobe")
    if ffprobe is None: raise RuntimeError("ffprobe nao encontrado. Instale o ffmpeg.")
    cmd = [
        ffprobe, "-v", "error",
        "-show_entries", "format=duration",
        "-of", "json",
        str(video_path),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    info = json.loads(result.stdout)
    return float(info["format"]["duration"])


def obter_frames_video(video_path, sample_fps=1, verbose=True):
    # extrai frames de um video com taxa de amostragem 
    # com ffmpeg diretamente via subprocess, lendo os frames em memoria
    # (sem gravar arquivos temporarios no disco).
    # sample_fps : frames por segundo a extrair.
    #               sample_fps=2   -> 2 frames por segundo
    #               sample_fps=0.5 -> 1 frame a cada 2 segundos
    # retorna 
    # frames    : lista de PIL.Image.Image no modo RGB.
    # timestamps: lista de floats com o tempo (em segundos) de cada frame.
    
    ffmpeg = shutil.which("ffmpeg")
    if ffmpeg is None:
        raise RuntimeError("ffmpeg nao encontrado no PATH.")

    video_path = str(video_path)
    if not os.path.isfile(video_path):
        raise FileNotFoundError(f"Video nao encontrado: {video_path}")

    if sample_fps <= 0:
        raise ValueError(f"sample_fps deve ser > 0. Recebido: {sample_fps}")

    # metadados do video
    duracao = _duracao_video(video_path)
    n_frames_esperado = max(1, int(duracao * sample_fps))
    if verbose:
        intervalo = 1.0 / sample_fps
        print(f"Video      : {video_path}")
        print(f"Duracao    : {duracao:.2f}s")
        print(f"sample_fps : {sample_fps} (1 frame a cada {intervalo:.2f}s)")
        print(f"Frames est.: ~{n_frames_esperado}")

    # Comando ffmpeg
    max_frames = n_frames_esperado + 10
    cmd = [
        ffmpeg, "-y", "-i", video_path, "-vf", f"fps={sample_fps}", "-f", "rawvideo",
        "-pix_fmt", "rgb24", "-vframes", str(max_frames), "pipe:1", ]

    # obtem dimensoes reais do video
    ffprobe_cmd = [
        shutil.which("ffprobe"), "-v", "error", "-select_streams", "v:0", 
        "-show_entries", "stream=width,height", "-of", "json", video_path,
    ]
    probe_out = subprocess.run(ffprobe_cmd, capture_output=True, text=True, check=True)
    probe_info = json.loads(probe_out.stdout)
    width  = probe_info["streams"][0]["width"]
    height = probe_info["streams"][0]["height"]
    frame_bytes = width * height * 3  # 3 bytes por pixel (RGB24)
    proc = subprocess.run(cmd, capture_output=True)
    raw_data = proc.stdout

    if len(raw_data) == 0:
        stderr_msg = proc.stderr.decode(errors="replace")[-500:]
        raise RuntimeError(f"ffmpeg nao retornou dados.\nstderr:\n{stderr_msg}")

    # frames individuais
    n_frames_real = len(raw_data) // frame_bytes
    frames = []
    timestamps = []
    intervalo_s = 1.0 / sample_fps
    pbar = tqdm(range(n_frames_real), desc="Convertendo frames", unit="frame",
                disable=not verbose)
    for idx in pbar:
        offset = idx * frame_bytes
        raw_frame = raw_data[offset: offset + frame_bytes]
        arr = np.frombuffer(raw_frame, dtype=np.uint8).reshape(height, width, 3)
        img = Image.fromarray(arr, mode="RGB")
        frames.append(img)
        timestamps.append(round(idx * intervalo_s, 4))

    return frames, timestamps




#### exemplo inferencia frames amostrados de um video

In [19]:
# labels do ImageNet para interpretar resultados
arq = open("labels_imagenet.txt", "r")
labels = arq.readlines()
labels = [l.lower().strip() for l in labels]
print(labels[0:5])


['tench', 'goldfish', 'great white shark', 'tiger shark', 'hammerhead']


In [20]:
VIDEO_PATH = "./data/video.mp4"
SAMPLE_FPS = 1.0   # 1 frame por segundo

if shutil.which("ffmpeg") is None:
    print("[AVISO] ffmpeg nao encontrado. Instale para habilitar esta funcionalidade.")
else:
    # extrai frames
    frames, timestamps = obter_frames_video(
        video_path=VIDEO_PATH, sample_fps=SAMPLE_FPS, verbose=True, )

    print(f"Frames extraidos : {len(frames)}")
    print(f"Timestamps (s)   : {timestamps[:5]} ...")


Video      : ./data/video.mp4
Duracao    : 29.80s
sample_fps : 1.0 (1 frame a cada 1.00s)
Frames est.: ~29


Convertendo frames:   0%|          | 0/30 [00:00<?, ?frame/s]

Frames extraidos : 30
Timestamps (s)   : [0.0, 1.0, 2.0, 3.0, 4.0] ...


In [21]:
# Predicao nos frames extraidos
inicio = time.time()
resultados_video = predicao_batch(model_effb0, frames, transform_inference,
    batch_size=16, top_k=3, verbose=False, )

print(f"duracao: {time.time()-inicio}")


duracao: 0.7080020904541016


In [22]:
print("Predicoes por frame:")
for ts, res in zip(timestamps[:10], resultados_video[:10]):
    idx_0 = res['top_k_idx'][0]
    print(f"  t={ts:6.2f}s  top-1 idx={idx_0} {labels[idx_0]:15s}  |  proba={res['top_k_proba'][0]:.4f}")


Predicoes por frame:
  t=  0.00s  top-1 idx=762 restaurant       |  proba=0.4435
  t=  1.00s  top-1 idx=762 restaurant       |  proba=0.3197
  t=  2.00s  top-1 idx=762 restaurant       |  proba=0.1975
  t=  3.00s  top-1 idx=678 neck brace       |  proba=0.1585
  t=  4.00s  top-1 idx=903 wig              |  proba=0.5738
  t=  5.00s  top-1 idx=903 wig              |  proba=0.8420
  t=  6.00s  top-1 idx=627 limousine        |  proba=0.0565
  t=  7.00s  top-1 idx=785 seat belt        |  proba=0.0841
  t=  8.00s  top-1 idx=582 grocery store    |  proba=0.0412
  t=  9.00s  top-1 idx=929 ice lolly        |  proba=0.1149


#### obter_imagens_diretorio — lista imagens de uma pasta

In [23]:
# Extensoes de imagem reconhecidas
_EXTENSOES_IMAGEM = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".tif", }

def obter_imagens_diretorio(diretorio, recursivo=False, extensoes=None, verbose=True,):
    # obtem imagens e seus caminhos a partir de um diretorio.
    if extensoes is None: extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".tif", }
    raiz = Path(diretorio)
    if not raiz.is_dir():
        raise NotADirectoryError(f"Diretorio nao encontrado: {diretorio}")
    # Coletar paths
    padrao = "**/*" if recursivo else "*"
    todos = sorted(raiz.glob(padrao))
    paths_filtrados = [str(p.resolve()) for p in todos if p.is_file() and p.suffix.lower() in extensoes]
    imagens = []
    paths_ok = []
    erros = 0
    pbar = tqdm(paths_filtrados, desc="Carregando imagens", unit="img",
                disable=not verbose)
    for path in pbar:
        try:
            img = Image.open(path).convert("RGB")
            imagens.append(img)
            paths_ok.append(path)
        except Exception as e:
            erros += 1
            pbar.write(f"[AVISO] Nao foi possivel abrir {path}: {e}")

    if verbose and erros:
        print(f"[AVISO] {erros} arquivo(s) nao puderam ser abertos e foram ignorados.")

    return imagens, paths_ok



#### exemplo inferencia imagens diretorio

In [24]:
DIRETORIO_IMAGENS = "./data"

if os.path.isdir(DIRETORIO_IMAGENS):
    imagens, paths = obter_imagens_diretorio(diretorio=DIRETORIO_IMAGENS,
        recursivo=False, verbose=True, )
else:
    print(f"[INFO] Diretorio '{DIRETORIO_IMAGENS}' nao encontrado.")


Carregando imagens:   0%|          | 0/9 [00:00<?, ?img/s]

In [25]:
resultados_dir = predicao_batch(model_effb0, imagens, transform_inference,
        batch_size=16, top_k=3, verbose=False, )

print("Resumo das predicoes:")
for path, res in zip(paths[:5], resultados_dir[:5]):
    nome = os.path.basename(path)
    idx_0 = res['top_k_idx'][0]
    print(f"  {nome:30s}  top-1 idx={idx_0} {labels[idx_0]:15s}  |  proba={res['top_k_proba'][0]:.4f}")
if len(paths) > 5:
    print(f"  ... e mais {len(paths)-5} imagens")


Resumo das predicoes:
  t1.jpg                          top-1 idx=762 restaurant       |  proba=0.1443
  t2.jpg                          top-1 idx=703 park bench       |  proba=0.0631
  t3.jpg                          top-1 idx=762 restaurant       |  proba=0.1476
  t4.jpg                          top-1 idx=860 tobacco shop     |  proba=0.1306
  t5.jpg                          top-1 idx=903 wig              |  proba=0.2801
  ... e mais 4 imagens
